In [ ]:
# Input: repeated_5x5_mean_oof_predictions.csv with Metaphlan_ID, Cohort, Target, Model, Observed_Log2, and Predicted_Log2_Mean.
# Input: repeated_5x5_best_models_summary.csv with Target and Model, one selected model per target.

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, t

arial_candidates = [
    Path(r"C:\Windows\Fonts\arial.ttf"),
    Path(r"C:\Windows\Fonts\Arial.ttf"),
]
arial_file = next((path for path in arial_candidates if path.exists()), None)
if arial_file is not None:
    font_manager.fontManager.addfont(str(arial_file))

try:
    resolved_arial = font_manager.findfont("Arial", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError(
        "Arial was not found. Install Arial or update arial_candidates."
    ) from exc

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})

print(f"Arial: {resolved_arial}")
print(f"Matplotlib PDF font type: {mpl.rcParams['pdf.fonttype']}")


In [ ]:
BASE_DIR = Path.cwd().resolve()
MEAN_OOF_FILENAME = "repeated_5x5_mean_oof_predictions.csv"
OUTPUT_DIR = BASE_DIR / "scatter_plots"
SHOW_FIGURES = True

COHORT_STYLES = {
    "CRC": {"color": "#E64B35"},
    "CXY": {"color": "#9F79B6"},
    "VD": {"color": "#00A087"},
    "T2D": {"color": "#3C5488"},
    "VAS": {"color": "#D07D07"},
    "Yangzixin": {"color": "#68AADC"},
}
COHORT_LABEL_MAP = {"cxy": "CXY", "va": "VAS", "Aging": "VD", "VD_Aging": "VD"}

TARGET_ORDER = ["TA", "IAM", "ILA", "I3C", "IE", "IAA", "IA", "IPA", "INDOLE"]

def locate_results_dir(start):
    start = Path(start).resolve()
    candidates = [p for p in [start / "Results", start] if (p / MEAN_OOF_FILENAME).is_file()]
    if len(candidates) != 1:
        raise FileNotFoundError(f"Expected one {MEAN_OOF_FILENAME} in the notebook directory or its Results subdirectory; found {len(candidates)}.")
    return candidates[0]

RESULTS_DIR = locate_results_dir(BASE_DIR)
mean_oof = pd.read_csv(RESULTS_DIR / MEAN_OOF_FILENAME, encoding="utf-8-sig")

required_prediction_columns = {
    "Metaphlan_ID",
    "Cohort",
    "Target",
    "Model",
    "Observed_Log2",
    "Predicted_Log2_Mean",
}
missing_prediction_columns = sorted(required_prediction_columns - set(mean_oof.columns))
if missing_prediction_columns:
    raise KeyError(f"Missing prediction columns: {missing_prediction_columns}")
if mean_oof.empty or mean_oof[["Target", "Model"]].isna().any().any():
    raise ValueError("Mean OOF data must contain non-missing target and model labels.")
selected_models = mean_oof[["Target", "Model"]].drop_duplicates()
if selected_models["Target"].duplicated().any():
    raise ValueError("Mean OOF data contain multiple models per target; provide predictions from the selected models only.")
selected_predictions = mean_oof.copy()
if selected_predictions.duplicated(["Target", "Metaphlan_ID"]).any():
    raise ValueError("The selected mean OOF data contain duplicated sample-target pairs.")
if set(selected_predictions["Target"]) != set(selected_models["Target"]):
    raise ValueError("At least one selected model has no mean OOF predictions.")

selected_predictions["Cohort"] = selected_predictions["Cohort"].replace(
    COHORT_LABEL_MAP
)
unknown_cohorts = sorted(set(selected_predictions["Cohort"]) - set(COHORT_STYLES))
if unknown_cohorts:
    raise ValueError(f"Undefined cohort styles: {unknown_cohorts}")

ordered_targets = [target for target in TARGET_ORDER if target in set(selected_models["Target"])]
ordered_targets.extend(
    target for target in selected_models["Target"] if target not in ordered_targets
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mean OOF input: {MEAN_OOF_FILENAME}")
print(f"Selected sample-target rows: {len(selected_predictions):,}")
print(f"Targets: {len(ordered_targets)}")
print("Output directory: scatter_plots")


In [ ]:
def ols_mean_confidence_interval(x, y, x_grid, confidence=0.95):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_grid = np.asarray(x_grid, dtype=float)
    n = x.size
    if n < 3:
        raise ValueError("At least three valid samples are required for OLS.")

    design = np.column_stack([np.ones(n), x])
    beta, *_ = np.linalg.lstsq(design, y, rcond=None)
    fitted = design @ beta
    residual = y - fitted
    dof = n - 2
    sxx = np.sum((x - x.mean()) ** 2)
    if sxx <= 0:
        raise ValueError("The x values have no variation; an OLS line cannot be fitted.")

    residual_variance = np.sum(residual ** 2) / dof
    y_grid = beta[0] + beta[1] * x_grid
    se_mean = np.sqrt(
        residual_variance * (1.0 / n + (x_grid - x.mean()) ** 2 / sxx)
    )
    critical = t.ppf(0.5 + confidence / 2.0, dof)
    return y_grid, y_grid - critical * se_mean, y_grid + critical * se_mean

def format_p_value(p_value):
    return "P < 0.001" if p_value < 0.001 else f"P = {p_value:.3f}"

def safe_filename(value):
    return "".join(char if char.isalnum() or char in "-_" else "_" for char in str(value))

def plot_one_target(frame, target, model_name, output_dir, order_number, show=True):
    pair = frame.loc[
        (frame["Target"] == target) & (frame["Model"] == model_name),
        ["Metaphlan_ID", "Cohort", "Observed_Log2", "Predicted_Log2_Mean"],
    ].copy()
    pair = pair.rename(
        columns={
            "Observed_Log2": "observed",
            "Predicted_Log2_Mean": "predicted",
        }
    )
    pair[["observed", "predicted"]] = (
        pair[["observed", "predicted"]]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )
    pair = pair.dropna(subset=["Cohort", "observed", "predicted"])
    if len(pair) < 3:
        raise ValueError(f"{target} has fewer than three valid paired samples.")

    x = pair["observed"].to_numpy(dtype=float)
    y = pair["predicted"].to_numpy(dtype=float)
    rho, p_value = spearmanr(x, y)

    x_span = np.ptp(x)
    x_padding = max(x_span * 0.05, 0.05)
    x_grid = np.linspace(x.min() - x_padding, x.max() + x_padding, 300)
    y_fit, y_low, y_high = ols_mean_confidence_interval(x, y, x_grid)

    mpl.rcParams.update({
        "font.family": "Arial",
        "font.sans-serif": ["Arial"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })
    fig, ax = plt.subplots(figsize=(11.7, 10.0))
    ax.set_axisbelow(True)
    ax.grid(True, color="#D7DDE2", linewidth=1.0, alpha=0.78)

    ax.fill_between(
        x_grid,
        y_low,
        y_high,
        color="#193E58",
        alpha=0.20,
        linewidth=0,
        zorder=1,
    )
    cohort_values = pair["Cohort"].to_numpy()
    for cohort, style in COHORT_STYLES.items():
        cohort_mask = cohort_values == cohort
        cohort_n = int(cohort_mask.sum())
        if cohort_n == 0:
            continue
        ax.scatter(
            x[cohort_mask],
            y[cohort_mask],
            s=200,
            color=style["color"],
            alpha=0.82,
            edgecolors="black",
            linewidths=0.6,
            label=f"{cohort} (n={cohort_n})",
            rasterized=True,
            zorder=2,
        )
    ax.plot(
        x_grid,
        y_fit,
        color="#04253C",
        linewidth=3.0,
        linestyle="--",
        zorder=3,
    )

    stats_text = (
        f"Spearman rho = {rho:.3f}\n"
        f"{format_p_value(p_value)}\n"
        f"N = {len(pair)}"
    )
    ax.text(
        0.04,
        0.96,
        stats_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=19,
        color="#222222",
        bbox={
            "boxstyle": "round,pad=0.42",
            "facecolor": "white",
            "edgecolor": "#C9CED3",
            "linewidth": 1.4,
            "alpha": 0.94,
        },
        zorder=4,
    )

    legend = ax.legend(
        title="Cohort",
        loc="lower right",
        fontsize=13,
        title_fontsize=14,
        markerscale=1,
        frameon=True,
        borderpad=0.65,
        labelspacing=0.45,
    )
    legend.get_frame().set_facecolor("white")
    legend.get_frame().set_edgecolor("#C9CED3")
    legend.get_frame().set_alpha(0.94)
    legend.set_zorder(4)

    ax.set_title(f"{target} - {model_name}", fontsize=26, fontweight="bold", pad=18)
    ax.set_xlabel("Observed value, log2(value + 1)", fontsize=20, labelpad=10)
    ax.set_ylabel("Mean OOF-predicted value, log2(value + 1)", fontsize=20, labelpad=10)
    ax.tick_params(axis="both", labelsize=17, width=1.6, length=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.6)
    ax.spines["bottom"].set_linewidth(1.6)
    ax.margins(y=0.05)

    output_name = (
        f"{order_number:02d}_{safe_filename(target)}_"
        f"{safe_filename(model_name)}_mean_oof_scatter.pdf"
    )
    output_path = output_dir / output_name
    fig.tight_layout()
    fig.savefig(
        output_path,
        format="pdf",
        dpi=300,
        metadata={
            "Title": f"{target} - {model_name}",
            "Subject": "Observed versus mean out-of-fold predicted log2 values",
            "Creator": "Matplotlib",
        },
    )

    if show:
        plt.show()
    plt.close(fig)

    return {
        "Target": target,
        "Model": model_name,
        "N": len(pair),
        "Spearman_rho": rho,
        "P_value": p_value,
        "PDF": output_path.name,
    }


In [ ]:
best_model_by_target = best_models.set_index("Target")["Model"].to_dict()
plot_results = [
    plot_one_target(
        selected_predictions,
        target=target,
        model_name=best_model_by_target[target],
        output_dir=OUTPUT_DIR,
        order_number=index,
        show=SHOW_FIGURES,
    )
    for index, target in enumerate(ordered_targets, start=1)
]

plot_result_table = pd.DataFrame(plot_results)
display(plot_result_table)
print("Generated PDFs:")
for filename in plot_result_table["PDF"]:
    print(filename)
